In [ ]:
# ===============================
# Microarray Predictive Analytics
# Fully Error-Proof Version
# ===============================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ===============================
# Step 1: Check if dataset exists
# ===============================

file_name = "/content/air_quality_dataset - air_quality_dataset.csv"

if not os.path.exists(file_name):
    print("Dataset not found! Creating a synthetic dataset...")

    np.random.seed(42)
    X_synthetic = np.random.rand(120, 200)   # 120 samples, 200 genes
    y_synthetic = np.random.randint(0, 2, 120)

    df_synthetic = pd.DataFrame(X_synthetic)
    df_synthetic['label'] = y_synthetic
    df_synthetic.to_csv(file_name, index=False)

    print("Synthetic dataset created as 'microarray.csv'\n")

# ===============================
# Step 2: Load dataset
# ===============================

data = pd.read_csv(file_name)

print("Dataset Loaded Successfully!")
print("Shape:", data.shape)

# ===============================
# Step 3: Clean column names
# ===============================

data.columns = data.columns.str.strip()

# ===============================
# Step 4: Separate features & label
# ===============================

# Safe approach: last column = target
X = data.iloc[:, :-1]
y = data.iloc[:, -1]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

# ===============================
# Step 5: Handle missing values
# ===============================

# Impute numerical columns with mean
for col in X.select_dtypes(include=np.number).columns:
    X[col] = X[col].fillna(X[col].mean())

# Impute categorical columns with mode
for col in X.select_dtypes(include='object').columns:
    X[col] = X[col].fillna(X[col].mode()[0])

# ===============================
# Step 6: Feature scaling
# ===============================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.select_dtypes(include=np.number)) # Only scale numeric features

# If X contains non-numeric columns that need to be part of the final feature set,
# they should be one-hot encoded or handled appropriately before scaling or feature selection.
# For now, we'll proceed with only numeric scaled features for feature selection.

# ===============================
# Step 7: Feature selection
# ===============================

k = min(50, X_scaled.shape[1])  # avoid error if features < 50
selector = SelectKBest(score_func=f_classif, k=k)
X_selected = selector.fit_transform(X_scaled, y)

print("Selected Features shape:", X_selected.shape)

# ===============================
# Step 8: Train-Test Split
# ===============================

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)

# ===============================
# Step 9: Model Training
# ===============================

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ===============================
# Step 10: Prediction
# ===============================

y_pred = model.predict(X_test)

# ===============================
# Step 11: Evaluation
# ===============================

print("\n===== RESULTS =====")
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n", classification_report(y_test, y_pred))

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))